# Feature Engineering

Cleaning + creating new features based on EDA findings.
Every decision here has a reason — not just running `.fillna(mean)` blindly.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

TARGET = 'SeriousDlqin2yrs'

In [2]:
df = pd.read_csv('../data/raw/cs-training.csv', index_col=0)
print('Raw shape:', df.shape)
df.head()

Raw shape: (150000, 11)


## Step 1 — Drop bad rows

age == 0 is clearly erroneous — just 1 row, safe to drop.

In [3]:
df = df[df['age'] > 0]
print('After dropping age==0:', df.shape)

After dropping age==0: (149999, 11)


## Step 2 — Fix RevolvingUtilization

This is a ratio — valid range is 0 to 1.
Values above 1 are data errors, cap them at 1.0.

In [4]:
print('Before — max:', df['RevolvingUtilizationOfUnsecuredLines'].max())
print('Before — values > 1:', (df['RevolvingUtilizationOfUnsecuredLines'] > 1).sum())

df['RevolvingUtilizationOfUnsecuredLines'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(upper=1.0)

print('\nAfter — max:', df['RevolvingUtilizationOfUnsecuredLines'].max())

Before — max: 50708.0
Before — values > 1: 3321

After — max: 1.0


## Step 3 — Fix past-due sentinel values

Values of 96 and 98 in the past-due columns are error codes, not real data.
Replace them with NaN, then fill with 0 (most people have 0 late payments).

In [5]:
past_due_cols = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate'
]

for col in past_due_cols:
    n_replaced = (df[col] >= 96).sum()
    df[col] = df[col].where(df[col] < 96, np.nan)
    df[col] = df[col].fillna(0)
    print(f'{col}: replaced {n_replaced} sentinel values')

# verify
df[past_due_cols].max()

NumberOfTime30-59DaysPastDueNotWorse: replaced 269 sentinel values
NumberOfTime60-89DaysPastDueNotWorse: replaced 269 sentinel values
NumberOfTimes90DaysLate: replaced 269 sentinel values


## Step 4 — Handle missing values

**MonthlyIncome (~20% missing):**  
Income correlates with age — using age-group median is smarter than global median.

**NumberOfDependents (~2.6% missing):**  
Small enough — simple median fill works fine.

In [6]:
# create age groups for groupby imputation
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 35, 45, 55, 65, 120],
    labels=['<25', '25-35', '35-45', '45-55', '55-65', '65+']
)

# check median income by age group (sanity check — should increase then decrease)
df.groupby('age_group', observed=True)['MonthlyIncome'].median()

In [7]:
# fill MonthlyIncome with age-group median
df['MonthlyIncome'] = df.groupby('age_group', observed=True)['MonthlyIncome'].transform(
    lambda x: x.fillna(x.median())
)

# fill NumberOfDependents with overall median
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())

# confirm no missing values left
print('Missing values remaining:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values remaining:
Series([], dtype: int64)


## Step 5 — Create new features

Domain-driven features — things that actually signal credit risk.

In [8]:
# total times ever past due across all windows
df['total_past_due'] = (
    df['NumberOfTime30-59DaysPastDueNotWorse'] +
    df['NumberOfTime60-89DaysPastDueNotWorse'] +
    df['NumberOfTimes90DaysLate']
)

# monthly debt payment estimate from debt ratio
# DebtRatio = monthly debt payments / monthly income
df['monthly_debt'] = df['DebtRatio'] * df['MonthlyIncome']

# income per dependent — proxy for financial stress
df['income_per_dependent'] = df['MonthlyIncome'] / (df['NumberOfDependents'] + 1)

# binary flag — has the person ever been late?
df['has_past_due'] = (df['total_past_due'] > 0).astype(int)

print('New features added:')
print(['total_past_due', 'monthly_debt', 'income_per_dependent', 'has_past_due'])
df[['total_past_due', 'monthly_debt', 'income_per_dependent', 'has_past_due']].describe().round(2)

## Step 6 — Drop helper columns and verify final shape

In [9]:
# drop age_group (was only needed for imputation — it's a string, not useful for models)
df = df.drop(columns=['age_group'])

print('Final shape:', df.shape)
print('Columns:', df.columns.tolist())
print('\nAny nulls?', df.isnull().sum().sum())

Final shape: (149999, 15)
Columns: ['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'total_past_due', 'monthly_debt', 'income_per_dependent', 'has_past_due']

Any nulls? 0


## Step 7 — Quick sanity check on new features vs target

In [10]:
# default rate by has_past_due — should be much higher for those with past dues
print('Default rate by past due history:')
print(df.groupby('has_past_due')[TARGET].mean().round(3))

Default rate by past due history:
has_past_due
0    0.028
1    0.220
Name: SeriousDlqin2yrs, dtype: float64


In [11]:
# correlation of new features with target
new_features = ['total_past_due', 'monthly_debt', 'income_per_dependent', 'has_past_due']
corr_with_target = df[new_features + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(ascending=False)
print('Correlation with target:')
print(corr_with_target.round(3))

Correlation with target:
total_past_due          0.389
has_past_due            0.307
monthly_debt           -0.008
income_per_dependent   -0.028
Name: SeriousDlqin2yrs, dtype: float64


## Step 8 — Save processed data

In [12]:
df.to_csv('../data/processed/loan_engineered.csv', index=False)
print(f'Saved → data/processed/loan_engineered.csv  {df.shape}')

Saved → data/processed/loan_engineered.csv  (149999, 15)


## Key decisions made here

- **Dropped 1 row** where age == 0 → 149,999 rows remaining
- **RevolvingUtilization** capped at 1.0 — was 50,708 max before; 3,321 bad values fixed
- **Sentinel values** (96/98) replaced in all 3 past-due columns — 269 rows each; max now 13 / 11 / 17
- **MonthlyIncome** filled with age-group median — income peaks at 45–55 ($6,267/month) and drops post-retirement ($4,968 for 65+), confirming the grouping made sense
- **0 nulls** remaining in final dataset (149,999 rows × 15 columns)

**New features — what actually worked:**
- `total_past_due` — correlation 0.389 with target, strongest signal
- `has_past_due` — correlation 0.307; default rate jumps from 2.8% → 22% once someone has any past due (8x difference)
- `monthly_debt` — correlation −0.008, basically noise (DebtRatio itself has extreme values that inflate this)
- `income_per_dependent` — correlation −0.028, too weak

`monthly_debt` and `income_per_dependent` will be dropped before modeling — they add noise, not signal.